# Reload saved splits, merge, resample, and save

This notebook:
1. Reads the CSV files produced by the previous notebook
2. Merges features with targets for each split
3. Resamples only the feature trajectories inside each `fault_type`
4. Rebuilds the target after resampling
5. Saves the result to `preprocessed_resampled`


In [1]:

import os
from os.path import join, exists
from pathlib import Path

import numpy as np
import pandas as pd


In [2]:

# =========================
# Configuration
# =========================

INPUT_DIR = r'/home/ilya_treyvish/projects/lbnl_fdd/data/processed/Simulated RTU'
OUTPUT_DIR = join(INPUT_DIR, 'preprocessed_resampled_1')

# Resampling options:
# - 'mean'      : average each consecutive block of K points
# - 'every_kth' : keep every K-th row
RESAMPLE_METHOD = 'mean'
RESAMPLE_K = 3

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print('INPUT_DIR :', INPUT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('METHOD    :', RESAMPLE_METHOD)
print('K         :', RESAMPLE_K)


INPUT_DIR : /home/ilya_treyvish/projects/lbnl_fdd/data/processed/Simulated RTU
OUTPUT_DIR: /home/ilya_treyvish/projects/lbnl_fdd/data/processed/Simulated RTU/preprocessed_resampled_1
METHOD    : mean
K         : 3


In [3]:

def _read_feature_df(path):
    df = pd.read_csv(path)

    unnamed = [c for c in df.columns if c.startswith('Unnamed:')]
    if unnamed:
        df = df.drop(columns=unnamed)

    if {'fault_type', 'Datetime'}.issubset(df.columns):
        df['Datetime'] = pd.to_datetime(df['Datetime'])
        df = df.set_index(['fault_type', 'Datetime']).sort_index()
    else:
        raise ValueError(f'Could not find fault_type/Datetime columns in {path}')

    return df


def _read_target_series(path):
    target = pd.read_csv(path)

    unnamed = [c for c in target.columns if c.startswith('Unnamed:')]
    if unnamed:
        target = target.drop(columns=unnamed)

    if 'target' not in target.columns:
        raise ValueError(f'Column "target" not found in {path}')

    if {'fault_type', 'Datetime'}.issubset(target.columns):
        target['Datetime'] = pd.to_datetime(target['Datetime'])
        target = target.set_index(['fault_type', 'Datetime']).sort_index()['target']
    else:
        # fallback in case target was saved with a plain index
        if target.shape[1] == 1:
            target = target.iloc[:, 0]
            target.name = 'target'
        else:
            target = target['target']

    return target


def read_split(split_name, input_dir=INPUT_DIR):
    df_path = join(input_dir, f'{split_name}_df.csv')
    target_path = join(input_dir, f'{split_name}_target.csv')

    features = _read_feature_df(df_path)
    target = _read_target_series(target_path)

    if isinstance(target.index, pd.MultiIndex):
        target = target.reindex(features.index)

    merged = features.copy()
    merged['target'] = target.values if len(target) == len(features) else target.reindex(features.index).values

    return features, target, merged


In [4]:

def downsample_feature_trajectories(df_features, method='mean', k=10):
    """
    Downsample features independently inside each fault_type trajectory.

    Parameters
    ----------
    df_features : pd.DataFrame
        MultiIndex DataFrame with index levels ['fault_type', 'Datetime'].
    method : str
        'mean'      -> average each consecutive block of k points
        'every_kth' -> keep every k-th row
    k : int
        Block size / step.
    """
    if method not in {'mean', 'every_kth'}:
        raise ValueError("method must be either 'mean' or 'every_kth'")

    def process_group(x):
        x = x.reset_index().sort_values('Datetime').reset_index(drop=True)
        feature_cols = [c for c in x.columns if c not in ['fault_type', 'Datetime']]

        if method == 'every_kth':
            out = x.iloc[::k].copy()
            return out[['fault_type', 'Datetime'] + feature_cols]

        grp = np.arange(len(x)) // k
        out = x.groupby(grp)[feature_cols].mean()
        out['Datetime'] = x.groupby(grp)['Datetime'].first().values
        out['fault_type'] = x['fault_type'].iloc[0]
        return out[['fault_type', 'Datetime'] + feature_cols]

    result = (
        df_features
        .groupby(level=0, group_keys=False)
        .apply(process_group)
        .set_index(['fault_type', 'Datetime'])
        .sort_index()
    )
    return result


def make_target_from_fault_type_index(df_features, original_target=None):
    """
    Rebuild target after downsampling.

    If original_target is provided, use the per-fault_type mapping from it.
    Otherwise fall back to target == fault_type.
    """
    if original_target is not None and isinstance(original_target.index, pd.MultiIndex):
        mapping = (
            pd.DataFrame({
                'fault_type': original_target.index.get_level_values('fault_type'),
                'target': original_target.values
            })
            .dropna()
            .drop_duplicates(subset=['fault_type'])
            .set_index('fault_type')['target']
            .to_dict()
        )

        target_values = df_features.index.get_level_values('fault_type').map(mapping)
        return pd.Series(target_values, index=df_features.index, name='target')

    return pd.Series(
        df_features.index.get_level_values('fault_type'),
        index=df_features.index,
        name='target'
    )


def resample_split(features, target, method=RESAMPLE_METHOD, k=RESAMPLE_K):
    features_resampled = downsample_feature_trajectories(features, method=method, k=k)
    target_resampled = make_target_from_fault_type_index(features_resampled, original_target=target)
    merged_resampled = features_resampled.copy()
    merged_resampled['target'] = target_resampled
    return features_resampled, target_resampled, merged_resampled


In [5]:

# Read splits saved by the previous notebook
train_df, train_target, train_merged = read_split('train')
val_df, val_target, val_merged = read_split('val')
test_df, test_target, test_merged = read_split('test')

print('Original shapes:')
print('train_df    ', train_df.shape, 'train_target    ', train_target.shape)
print('val_df      ', val_df.shape, 'val_target      ', val_target.shape)
print('test_df     ', test_df.shape, 'test_target     ', test_target.shape)


Original shapes:
train_df     (2050500, 23) train_target     (2050500,)
val_df       (720000, 23) val_target       (720000,)
test_df      (828025, 23) test_target      (828025,)


In [6]:

# Optional: inspect merged tables
train_merged.head()


RTU_RA_HUM  RTU_TOT_WATT  RTU_SEN_CAPA  \
fault_type Datetime                                                      
0          2018-07-20 01:00:00   50.000000     2192.1084    14631.9375   
           2018-07-20 01:01:00   59.898830     2183.5269    12823.3100   
           2018-07-20 01:02:00   59.870330     2184.2952    12819.8160   
           2018-07-20 01:03:00   59.842117     2185.8610    12811.6670   
           2018-07-20 01:04:00   59.814423     2187.7268    12801.4100   

                                RTU_SA_FAN_WATT  RTU_REFG_SUCT_TEMP  \
fault_type Datetime                                                   
0          2018-07-20 01:00:00         94.42871           53.074745   
           2018-07-20 01:01:00         94.42871           54.300545   
           2018-07-20 01:02:00         94.42871           53.771553   
           2018-07-20 01:03:00         94.42871           53.418230   
           2018-07-20 01:04:00         94.42871           53.139015   

                                RTU_SA_TEMP  RTU_REFG_DISC_TEMP  \
fault_type Datetime                                               
0          2018-07-20 01:00:00    52.272797           112.44799   
           2018-07-20 01:01:00    53.288155           112.72864   
           2018-07-20 01:02:00    53.024975           112.28627   
           2018-07-20 01:03:00    52.775420           112.12219   
           2018-07-20 01:04:00    52.532455           112.07385   

                                RTU_REFG_DISC_PRES  RTU_OA_FLOW     ZA_HUM  \
fault_type Datetime                                                          
0          2018-07-20 01:00:00          25093532.0    181.93083  59.926773   
           2018-07-20 01:01:00          25198948.0    181.93083  59.898830   
           2018-07-20 01:02:00          25183480.0    181.93083  59.870330   
           2018-07-20 01:03:00          25158652.0    181.93083  59.842117   
           2018-07-20 01:04:00          25130132.0    181.93083  59.814423   

                                ...  RTU_OA_TEMP  RTU_REFG_SUCT_PRES  \
fault_type Datetime             ...                                    
0          2018-07-20 01:00:00  ...    55.040030          14253456.0   
           2018-07-20 01:01:00  ...    55.033990          14454670.0   
           2018-07-20 01:02:00  ...    55.028000          14428462.0   
           2018-07-20 01:03:00  ...    55.022015          14384741.0   
           2018-07-20 01:04:00  ...    55.016030          14334126.0   

                                  ZA_TEMP  RTU_RA_TEMP  RTU_TOT_CAPA  \
fault_type Datetime                                                    
0          2018-07-20 01:00:00  73.611046    75.200000     17422.904   
           2018-07-20 01:01:00  73.353580    73.353580     17655.139   
           2018-07-20 01:02:00  73.096176    73.096176     17625.334   
           2018-07-20 01:03:00  72.839096    72.839096     17576.230   
           2018-07-20 01:04:00  72.583115    72.583115     17519.484   

                                RTU_OA_HUM  RTU_SA_FLOW  RTU_STG_STA  \
fault_type Datetime                                                    
0          2018-07-20 01:00:00   49.361565    2941.1106          1.0   
           2018-07-20 01:01:00   59.133976    2941.1106          1.0   
           2018-07-20 01:02:00   59.105840    2941.1106          1.0   
           2018-07-20 01:03:00   59.077980    2941.1106          1.0   
           2018-07-20 01:04:00   59.050636    2941.1106          1.0   

                                RTU_COMP_WATT  target  
fault_type Datetime                                    
0          2018-07-20 01:00:00      2097.6797       0  
           2018-07-20 01:01:00      2089.0981       0  
           2018-07-20 01:02:00      2089.8665       0  
           2018-07-20 01:03:00      2091.4324       0  
           2018-07-20 01:04:00      2093.2980       0  

[5 rows x 24 columns]

In [7]:

# Resample each split
train_df_resampled, train_target_resampled, train_merged_resampled = resample_split(
    train_df, train_target, method=RESAMPLE_METHOD, k=RESAMPLE_K
)

val_df_resampled, val_target_resampled, val_merged_resampled = resample_split(
    val_df, val_target, method=RESAMPLE_METHOD, k=RESAMPLE_K
)

test_df_resampled, test_target_resampled, test_merged_resampled = resample_split(
    test_df, test_target, method=RESAMPLE_METHOD, k=RESAMPLE_K
)

print('Resampled shapes:')
print('train_df    ', train_df_resampled.shape, 'train_target    ', train_target_resampled.shape)
print('val_df      ', val_df_resampled.shape, 'val_target      ', val_target_resampled.shape)
print('test_df     ', test_df_resampled.shape, 'test_target     ', test_target_resampled.shape)


Resampled shapes:
train_df     (683500, 23) train_target     (683500,)
val_df       (240000, 23) val_target       (240000,)
test_df      (276025, 23) test_target      (276025,)


In [8]:

# Save in the same split-wise format as before
train_df_resampled.to_csv(join(OUTPUT_DIR, 'train_df.csv'))
train_target_resampled.to_csv(join(OUTPUT_DIR, 'train_target.csv'))

val_df_resampled.to_csv(join(OUTPUT_DIR, 'val_df.csv'))
val_target_resampled.to_csv(join(OUTPUT_DIR, 'val_target.csv'))

test_df_resampled.to_csv(join(OUTPUT_DIR, 'test_df.csv'))
test_target_resampled.to_csv(join(OUTPUT_DIR, 'test_target.csv'))

# Also save merged versions for convenience
train_merged_resampled.to_csv(join(OUTPUT_DIR, 'train_merged.csv'))
val_merged_resampled.to_csv(join(OUTPUT_DIR, 'val_merged.csv'))
test_merged_resampled.to_csv(join(OUTPUT_DIR, 'test_merged.csv'))

print('Saved files to:', OUTPUT_DIR)
print(sorted(os.listdir(OUTPUT_DIR)))


Saved files to: /home/ilya_treyvish/projects/lbnl_fdd/data/processed/Simulated RTU/preprocessed_resampled_1
['test_df.csv', 'test_merged.csv', 'test_target.csv', 'train_df.csv', 'train_merged.csv', 'train_target.csv', 'val_df.csv', 'val_merged.csv', 'val_target.csv']


In [10]:

# Optional: if full-year files exist, resample them too
full_df_path = join(INPUT_DIR, 'full_year_data_downsampled_df.csv')
full_target_path = join(INPUT_DIR, 'full_year_data_downsampled_target.csv')

if exists(full_df_path) and exists(full_target_path):
    full_df = _read_feature_df(full_df_path)
    full_target = _read_target_series(full_target_path)

    full_df_resampled, full_target_resampled, full_merged_resampled = resample_split(
        full_df, full_target, method=RESAMPLE_METHOD, k=RESAMPLE_K
    )

    full_df_resampled.to_csv(join(OUTPUT_DIR, 'full_year_data_resampled_df.csv'))
    full_target_resampled.to_csv(join(OUTPUT_DIR, 'full_year_data_resampled_target.csv'))
    full_merged_resampled.to_csv(join(OUTPUT_DIR, 'full_year_data_resampled_merged.csv'))

    print('Full-year files also resampled and saved.')
    print(full_df_resampled.shape, full_target_resampled.shape)
else:
    print('Full-year downsampled files not found, skipping optional full-year block.')


Full-year downsampled files not found, skipping optional full-year block.
